In [1]:
import os 
import pandas as pd
from Bio import SeqIO
import re

In [2]:
def limit_length(bed_file,Min_Length=50, Max_Length=10000):
    df_bed = pd.read_csv(bed_file, sep="\t")
    df_bed["Length"] = df_bed["end"] - df_bed["start"] + 1
    df_limit_bed = df_bed[(Min_Length <= df_bed["Length"]) & (df_bed["Length"] <= Max_Length)]
    return df_limit_bed

In [3]:
def limit_split_disc(df_limit_bed, split_num=1, disc_num=1):
    df_limit_sd = df_limit_bed[(df_limit_bed["discordants"] >= disc_num) & (df_limit_bed["soft-clipped"] >= split_num)]
    return df_limit_sd

In [4]:
def limit_score(df_limit_bed, score=5):
    df_limit_score = df_limit_bed[df_limit_bed["score"] >= score]
    return df_limit_score

In [5]:
def limit_N(sequence, N_num=5):
    return sequence.count('N') <= N_num

In [6]:
def limit_simple_repeat(sequence, min_period=2, max_period=4, threshold=0.8):
    seq_upper = sequence.upper()
    for k in range(min_period, max_period + 1):
        pattern = r'(.{' + str(k) + r'})\1+'
        matches = re.finditer(pattern, seq_upper)
        total_repeat_len = 0
        for match in matches:
            total_repeat_len += len(match.group(0))
        if total_repeat_len / len(sequence) >= threshold:
            return True           
    return False

In [7]:
def limit_chr(low=1, high=12):
    chrom = [f"chr{i}" for i in range(low, high + 1)]
    return chrom

In [8]:
def extract_sequences(bed_file,genome_file):
    dict_genome = SeqIO.to_dict(SeqIO.parse(genome_file, "fasta"))
    df_bed_length = limit_length(bed_file)
    df_bed_disc_split = limit_split_disc(df_bed_length)
    df_bed_score = limit_score(df_bed_disc_split)
    extracted_data = []
    for _, row in df_bed_score.iterrows():
        chrom = str(row['chrom'])
        if chrom not in limit_chr():
            continue
        start = int(row['start'])
        end = int(row['end'])
        record = dict_genome[chrom]
        seq_length = len(record.seq)
        if start < 0:
            start = 0
        if end > seq_length:
            end = seq_length   
        if start >= end:
            print(f"start >= end: {chrom}:{start}-{end}")
            continue
        sequence = str(record.seq[start:end])
        if not limit_N(sequence):
            continue
        if limit_simple_repeat(sequence):
            continue
        location_str = f"{chrom}:{start}-{end}"
        extracted_data.append([location_str, sequence])
    df_limit_sequence = pd.DataFrame(extracted_data, columns=['location', 'sequence'])
    return df_limit_sequence

In [10]:
extract_sequences("/public3/labmember/zhouwm/shuidao/circleseq2/ERR10889838_osat/circle.bed","/public3/labmember/zhouwm/shuidao/genome/Osat/genome.fa")

,location,sequence
0,chr1:828535-829087,CTGCTCCATGCAAGCGAGCGCGACTTTAAGGTCGCCACGGACAGAG...
1,chr1:1446261-1446571,CCCGGCGCGGGGTGGTGCCGACTGGACCTCGCCCGGCGCGGGTGGC...
2,chr1:7791088-7791511,GGCGTCACGCCCGAGCGCTTCCTTCGGGGGGGGGGGGGGGCGATCG...
3,chr1:9077553-9077833,AAATCGGAGATCTCTCAAAACCACATCCGGCATGGCAGCAACAGTT...
4,chr1:10150737-10151039,GTAATACAATTGAAAGTATGTGTATGATTTTTTATAGATTTTTTGG...
...,...,...
271,chr9:16356771-16357181,CCTCGTCGTTGTCTCGTCGTGCGTGCGGCTCGTCTCCGGCGCCGGC...
272,chr9:17815911-17816044,CCTGGTGCGGCCTTTCGTTCAACGCGATGGTGTCGGCGGGCGTGCA...
273,chr9:18473208-18473527,GCCCGGTCGTCGCTGTTGTCCGAGTCGGCGCCGCCGGCGCCCACCG...
274,chr9:21189912-21190091,GATTGAGGTCAGGGCACTTGAGTGGCTTCCCATGGCCCCAATCGGC...


In [11]:
ara_dat = extract_sequences("/public3/labmember/zhouwm/shuidao/test/test/log/data/Ara_data/ara_all_circle.bed","/public3/labmember/zhouwm/shuidao/genome/Atha/genome.fa")

In [21]:
ara_dat

,sequences,labels
0,AATTTGGCCTTTACCATAATTTTTTTCACCGCCTTCAGTGCCGTAG...,1
1,GTGCTGTGGAAGCTGTATTGGCATGTATTGTTCTTATACGAGATTG...,1
2,GTATTGTTCTTATACGAGATTGTGTGTAATTGGTGGTACTGGCTCC...,1
3,CTGTTGCAGAGCCCATCTTGTTTAGAGTGTAATCACCCGAGGTTGC...,1
4,TGTCTCTTGACGTTCACACTCAATTCCTTATCACTTTCACCTCTGA...,1
...,...,...
134765,ATGGTGAAGAGTGAATCTGCGAAGTGGGTTTGAGAGAGGAGAAGAG...,1
134766,TAAAAAGATTGTGCTTACTTTTTAAAGTATATTTGCTCCTCAACAC...,1
134767,GATAGGCAGTCAAATTCTTGTGAGTCGTTTGGGGCTTGGTCGGAAT...,1
134768,AGAGACAGATGCCGATGTTCCGGTGAGTGTTGTACTGAATACCTTT...,1


In [22]:
ara_nag = pd.read_csv("/public3/labmember/zhouwm/shuidao/test/test/log/data/Ara_data/ara_train.neg.csv", sep=",", header=None, names=["sequences", "labels"])
ara_dat["labels"] = "1"
ara_dat_sample = ara_dat.sample(n=30000, random_state=61)
ara_dat_sample

,sequences,labels
88483,AAGAGACAGAAGAGGAAAGTGAAATAGGGTTTTGTTCATCTTCATC...,1
31272,CGGCGAGGTAGCTTCTTGCGTAGTTTAAAGGACAGTCTAGTGCGGA...,1
46113,GATCGAGCTCAGTGACATAGCATTTGCGGTAACTTCCGTCACCACT...,1
126291,AAGAGACATAGATTTATGTTGATTTTTGTAGCTGTTGATTTAGTTG...,1
70417,CATTAGGAGCTAAGGGCGTCCTCATAAACATTGGCCGTGGACCACA...,1
...,...,...
81653,GAAAGGAAAGGACCAGTCCTGACTATGGTCGAAGACGTAGCCCAAG...,1
22440,AGAGACAGCTCGGATATTGAACCGAGATATCAATAGGGAGCGTGCG...,1
387,CTGTCTCTAATTCTCTTCCCTTTTTCTAAAACCCATTAGTTTCCAA...,1
49566,TGTCTCTTATCAAACATTTGATTTTTTTCTAGTTTTATAAAGAACC...,1


In [23]:
dfs = pd.concat([ara_dat_sample,ara_nag])
dfs

,sequences,labels
88483,AAGAGACAGAAGAGGAAAGTGAAATAGGGTTTTGTTCATCTTCATC...,1
31272,CGGCGAGGTAGCTTCTTGCGTAGTTTAAAGGACAGTCTAGTGCGGA...,1
46113,GATCGAGCTCAGTGACATAGCATTTGCGGTAACTTCCGTCACCACT...,1
126291,AAGAGACATAGATTTATGTTGATTTTTGTAGCTGTTGATTTAGTTG...,1
70417,CATTAGGAGCTAAGGGCGTCCTCATAAACATTGGCCGTGGACCACA...,1
...,...,...
29995,ATGGAGTCCAGTGAGGATGTGGAGAATCTGAGTAGAGCTATTGAGA...,0
29996,ATGTGTGGGATTCTCGCTGTTCTTGGTTGCATCGACAACTCTCAAG...,0
29997,ATGGGGAAATACTCTGTTTTGATGATCTTTATCGCTTCTTCTCTTC...,0
29998,ATGGGGAAATACTCTGTTTTGATGATCTTTATCGCTTCTTCTCTTC...,0


In [24]:
from sklearn.utils import shuffle

dfs = shuffle(dfs, random_state=61)
dfs

,sequences,labels
75350,CTGTCTCTTATCCGAGGACGCGTCGTTCTAAGCAGGAGATCCATGC...,1
8527,ATGGGTCTTTGTTGGGGATCTCCATCTGATTCTCCTCCAACAACAA...,0
2783,CTGTCTCTAAGCTCATTTAACAGGTCCACCTTAATTTATGCCTGTG...,1
23093,ATGATGCACATGGACGAATCAAGAAAGGAAATCATTCGAGAAGAAG...,0
20036,ATGGTGAGGCAACAAAGAGCCTCTAAGGTTCATGAAGATAGACATC...,0
...,...,...
17535,TTTATGCAATAACTCAATTGTAATTATTCACAAATATAAATAAGTG...,1
1919,ATGAAGAAATTCTATCAAAATGCTGATCATGAGACTGATAAATCCA...,0
29408,ATGGGGTCTGTTTGCTGTGTGGCTGCAAAGGACAGAAATGTTCCCA...,0
15676,GCATACATCGCTCCCCATTGTATTCTTTACGCTGATTATGGGAAAA...,1


In [25]:
dfs.to_csv("/public3/labmember/zhouwm/shuidao/test/test/log/data/Ara_data/Ara.circle.filter.data.csv", index=False)